# Notebook 07 — Robustness to UNSEEN corruptions (spectral niche test)

Self-contained, single seed (42). Trains 4 models and evaluates on corruptions
that are **outside the training/augmentation grid**:

| Model | trained with |
|---|---|
| ViT-only | clean only |
| ViT + aug | noise-aware aug (JPEG/GaussNoise/GaussBlur/Downscale) |
| Fusion (ViT+spectral-CNN) | clean only |
| Fusion + aug | spectral + the same aug |

**Unseen corruptions:** salt&pepper, Poisson, gamma, contrast, motion-blur.

**Hypothesis.** noise-aug learns the 4 grid corruptions; on *unseen* types it may
transfer poorly. The spectral branch (architectural) may add robustness that aug
cannot — if so, the spectral stream has a genuine niche. All numbers computed at
run time; ~40-60 min on a T4 (4 trainings + eval).

In [1]:
# CELL 1: IMPORTS
import os, io, gc, random, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import cv2
from PIL import Image
from datasets import load_dataset
from transformers import ViTModel, ViTImageProcessor
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm.auto import tqdm
SEED=42
def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    os.environ['PYTHONHASHSEED']=str(s); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; print('Device:',DEVICE)

Device: cuda


In [2]:
# CELL 2: CONFIG
def free_mem():
    gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()
CONFIG=dict(hf='dragonintelligence/CIFAKE-image-dataset', model='google/vit-base-patch16-224-in21k',
            n_per_class=4000, test_ratio=0.15, val_ratio=0.15, train_ratio=0.70, seed=SEED,
            batch=32, ebatch=64, epochs=5, lr_vit=1e-5, lr_branch=1e-4, wd=0.05, feat=256,
            # TRAINING grid (for noise-aug) -- same as NB02/04
            train_grid={'JPEG':[90,70,50,30],'Noise':[5,10,20,30],'Blur':[3,5,7],'Downscale':[0.75,0.5,0.25]},
            out='/content/nb07_out')
os.makedirs(CONFIG['out'],exist_ok=True)

In [3]:
# CELL 3: LOAD + SAME SUBSET/SPLIT AS NB01
raw=load_dataset(CONFIG['hf']); lf=raw['train'].features['label']
FAKE_ID=lf.names.index('FAKE'); REAL_ID=lf.names.index('REAL')
pool={FAKE_ID:[],REAL_ID:[]}
for sp in ['train','test']:
    for it in tqdm(raw[sp],desc=sp,leave=False): pool[it['label']].append(it['image'])
rng=np.random.default_rng(SEED); n=CONFIG['n_per_class']
fi=rng.choice(len(pool[FAKE_ID]),n,replace=False); ri=rng.choice(len(pool[REAL_ID]),n,replace=False)
imgs=[pool[FAKE_ID][i] for i in fi]+[pool[REAL_ID][i] for i in ri]; lbl=[FAKE_ID]*n+[REAL_ID]*n
del pool; gc.collect()
itv,ite,ytv,yte=train_test_split(imgs,lbl,test_size=CONFIG['test_ratio'],stratify=lbl,random_state=SEED)
vf=CONFIG['val_ratio']/(CONFIG['train_ratio']+CONFIG['val_ratio'])
itr,iva,ytr,yva=train_test_split(itv,ytv,test_size=vf,stratify=ytv,random_state=SEED)
ytr=np.array(ytr); yva=np.array(yva); yte=np.array(yte)
print('train',len(itr),'val',len(iva),'test',len(ite))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/488 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/42.1M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/8.41M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20000 [00:00<?, ? examples/s]

train:   0%|          | 0/100000 [00:00<?, ?it/s]

test:   0%|          | 0/20000 [00:00<?, ?it/s]

train 5600 val 1200 test 1200


In [4]:
# CELL 4: TRAINING-GRID DEGRADERS (for noise-aug) + SPECTRAL FEATURE
def d_jpeg(p,q):
    b=io.BytesIO(); p.save(b,'JPEG',quality=int(q)); b.seek(0); return Image.open(b).convert('RGB').copy()
def d_noise(p,s,rng=None):
    a=np.asarray(p).astype(np.float32); r=rng or np.random.RandomState(0)
    return Image.fromarray(np.clip(a+r.normal(0,s,a.shape),0,255).astype(np.uint8))
def d_blur(p,k):
    k=int(k); k+=(k%2==0); return Image.fromarray(cv2.GaussianBlur(np.asarray(p),(k,k),0))
def d_down(p,sc):
    w,h=p.size; return p.resize((max(1,int(w*sc)),max(1,int(h*sc))),Image.BILINEAR).resize((w,h),Image.BILINEAR)
GRID={'JPEG':d_jpeg,'Noise':d_noise,'Blur':d_blur,'Downscale':d_down}
def random_degrade(p,rng):
    t=rng.choice(list(CONFIG['train_grid'].keys())); lv=rng.choice(CONFIG['train_grid'][t])
    return d_noise(p,lv,rng) if t=='Noise' else GRID[t](p,lv)

def gray32(p): return np.asarray(p.convert('L').resize((32,32),Image.BILINEAR),np.float32)/255.
def spec2d_one(p):
    m=np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(gray32(p))))).astype(np.float32)
    return ((m-m.mean())/(m.std()+1e-6))[None]  # (1,32,32)

In [5]:
# CELL 5: UNSEEN corruptions (NOT in the training grid)
def c_saltpepper(p, amt):
    a=np.asarray(p).copy(); r=np.random.RandomState(0); m=r.rand(*a.shape[:2])
    a[m<amt/2]=0; a[m>1-amt/2]=255; return Image.fromarray(a)
def c_poisson(p, scale):
    a=np.asarray(p).astype(np.float32); r=np.random.RandomState(0)
    lam=np.maximum(a/255.*scale,1e-6); noisy=r.poisson(lam)/scale*255.
    return Image.fromarray(np.clip(noisy,0,255).astype(np.uint8))
def c_gamma(p, g):
    a=np.asarray(p).astype(np.float32)/255.; return Image.fromarray((np.clip(a**g,0,1)*255).astype(np.uint8))
def c_contrast(p, f):
    a=np.asarray(p).astype(np.float32); mu=a.mean(); return Image.fromarray(np.clip((a-mu)*f+mu,0,255).astype(np.uint8))
def c_motion(p, k):
    k=int(k); ker=np.zeros((k,k),np.float32); ker[k//2,:]=1.0/k
    return Image.fromarray(cv2.filter2D(np.asarray(p),-1,ker))
UNSEEN={'SaltPepper':(c_saltpepper,[0.02,0.05,0.10]),
        'Poisson'   :(c_poisson,[60,30,15]),
        'Gamma'     :(c_gamma,[1.5,2.0,2.5]),
        'Contrast'  :(c_contrast,[0.6,0.4]),
        'MotionBlur':(c_motion,[5,9,13])}
print('Unseen corruption types:', list(UNSEEN.keys()))

Unseen corruption types: ['SaltPepper', 'Poisson', 'Gamma', 'Contrast', 'MotionBlur']


In [6]:
# CELL 6: MODELS (spectral-CNN branch + two-stream)
class SpectralCNN2D(nn.Module):
    def __init__(self,feat):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(64,feat),nn.ReLU())
    def forward(self,x): return self.net(x)
class TwoStream(nn.Module):
    def __init__(self,src,use_vit=True,use_spec=True,feat=256):
        super().__init__(); self.use_vit=use_vit; self.use_spec=use_spec; d=0
        if use_vit: self.vit=ViTModel.from_pretrained(src); d+=768
        if use_spec: self.branch=SpectralCNN2D(feat); d+=feat
        self.head=nn.Linear(d,2)
    def forward(self,pixel_values=None,spec=None):
        f=[]
        if self.use_vit: f.append(self.vit(pixel_values=pixel_values).last_hidden_state[:,0])
        if self.use_spec: f.append(self.branch(spec))
        return self.head(torch.cat(f,1))
def pgroups(m):
    vit=[p for nm,p in m.named_parameters() if nm.startswith('vit.')]
    oth=[p for nm,p in m.named_parameters() if not nm.startswith('vit.')]
    g=[]
    if vit: g.append({'params':vit,'lr':CONFIG['lr_vit']})
    if oth: g.append({'params':oth,'lr':CONFIG['lr_branch']})
    return g

In [7]:
# CELL 7: DATASET (optional aug; computes spectrum on the (possibly augmented) image)
proc=ViTImageProcessor.from_pretrained(CONFIG['model'])
clean_tf=Compose([Resize((224,224)),ToTensor(),Normalize(mean=proc.image_mean,std=proc.image_std)])
class DS(Dataset):
    def __init__(self,imgs,labels,augment,need_spec,seed=0):
        self.imgs=imgs; self.labels=labels; self.augment=augment; self.need_spec=need_spec
        self.rng=np.random.RandomState(seed)
    def __len__(self): return len(self.imgs)
    def __getitem__(self,i):
        im=self.imgs[i].convert('RGB')
        if self.augment: im=random_degrade(im,self.rng)
        pv=clean_tf(im)
        sp=torch.tensor(spec2d_one(im)) if self.need_spec else torch.zeros(1)
        return pv,sp,int(self.labels[i])
def loaders(augment,need_spec,seed):
    g=torch.Generator(); g.manual_seed(seed)
    tr=DataLoader(DS(itr,ytr,augment,need_spec,seed),batch_size=CONFIG['batch'],shuffle=True,generator=g)
    va=DataLoader(DS(iva,yva,False,need_spec),batch_size=CONFIG['ebatch'],shuffle=False)
    return tr,va

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [8]:
# CELL 8: TRAIN + PREDICT HELPERS
def train_one(use_vit,use_spec,augment,tag,seed=42):
    free_mem(); set_seed(seed)
    model=TwoStream(CONFIG['model'],use_vit,use_spec,CONFIG['feat']).to(DEVICE)
    tr,va=loaders(augment,use_spec,seed)
    opt=torch.optim.AdamW(pgroups(model),weight_decay=CONFIG['wd'])
    scaler=torch.amp.GradScaler('cuda',enabled=DEVICE=='cuda'); crit=nn.CrossEntropyLoss()
    best,bs=-1,None
    for ep in range(CONFIG['epochs']):
        model.train()
        for pv,sp,lb in tqdm(tr,desc=f'{tag} ep{ep+1}',leave=False):
            pv=pv.to(DEVICE) if use_vit else None; sp=sp.to(DEVICE) if use_spec else None; lb=lb.to(DEVICE)
            opt.zero_grad()
            with torch.autocast('cuda',enabled=DEVICE=='cuda'):
                loss=crit(model(pixel_values=pv,spec=sp),lb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        model.eval(); P,T=[],[]
        with torch.no_grad():
            for pv,sp,lb in va:
                pv=pv.to(DEVICE) if use_vit else None; sp=sp.to(DEVICE) if use_spec else None
                P.append(model(pixel_values=pv,spec=sp).argmax(1).cpu().numpy()); T.append(lb.numpy())
        f1=f1_score(np.concatenate(T),np.concatenate(P),pos_label=FAKE_ID)*100
        if f1>best: best=f1; bs={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(bs); print(f'  {tag}: best val F1={best:.2f}%'); return model

@torch.no_grad()
def predict(model,imgs):
    model.eval(); preds=[]; bs=CONFIG['ebatch']
    for s in range(0,len(imgs),bs):
        ch=imgs[s:s+bs]
        pv=torch.stack([clean_tf(im.convert('RGB')) for im in ch]).to(DEVICE) if model.use_vit else None
        sp=torch.stack([torch.tensor(spec2d_one(im)) for im in ch]).to(DEVICE) if model.use_spec else None
        preds.append(model(pixel_values=pv,spec=sp).argmax(1).cpu().numpy())
    return np.concatenate(preds)
def acc_on(model,imgs): return accuracy_score(yte,predict(model,imgs))*100

In [9]:
# CELL 9: TRAIN THE 4 MODELS (seed 42)
TRAIN_FUSION_AUG=True   # set False to skip best-of-both and save ~12 min
models={}
models['ViT-only']    = train_one(use_vit=True, use_spec=False, augment=False, tag='ViT-only')
models['ViT+aug']     = train_one(use_vit=True, use_spec=False, augment=True,  tag='ViT+aug')
models['Fusion']      = train_one(use_vit=True, use_spec=True,  augment=False, tag='Fusion')
if TRAIN_FUSION_AUG:
    models['Fusion+aug'] = train_one(use_vit=True, use_spec=True, augment=True, tag='Fusion+aug')
print('Clean test accuracy:')
for k,m in models.items(): print(f'  {k:12s}: {acc_on(m,ite):.2f}%')

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViT-only ep1:   0%|          | 0/175 [00:00<?, ?it/s]

ViT-only ep2:   0%|          | 0/175 [00:00<?, ?it/s]

...

Fusion+aug ep2:   0%|          | 0/175 [00:00<?, ?it/s]

Fusion+aug ep3:   0%|          | 0/175 [00:00<?, ?it/s]

Fusion+aug ep4:   0%|          | 0/175 [00:00<?, ?it/s]

Fusion+aug ep5:   0%|          | 0/175 [00:00<?, ?it/s]

  Fusion+aug: best val F1=95.39%
Clean test accuracy:
  ViT-only    : 97.25%
  ViT+aug     : 95.92%
  Fusion      : 97.25%
  Fusion+aug  : 94.83%


In [10]:
# CELL 10: EVALUATE ON UNSEEN CORRUPTIONS
rows=[]
for k,m in models.items():
    rows.append({'model':k,'corruption':'Clean','level':0,'acc':acc_on(m,ite)})
for cname,(fn,levels) in UNSEEN.items():
    for lv in tqdm(levels,desc=cname):
        deg=[fn(im,lv) for im in ite]
        for k,m in models.items():
            rows.append({'model':k,'corruption':cname,'level':lv,'acc':acc_on(m,deg)})
df=pd.DataFrame(rows)
piv=df.pivot_table(index=['corruption','level'],columns='model',values='acc')
order=[c for c in ['ViT-only','ViT+aug','Fusion','Fusion+aug'] if c in piv.columns]
piv=piv[order]
print(piv.round(2).to_string())
df.to_csv(os.path.join(CONFIG['out'],'table_unseen.csv'),index=False)

SaltPepper:   0%|          | 0/3 [00:00<?, ?it/s]

Poisson:   0%|          | 0/3 [00:00<?, ?it/s]

Gamma:   0%|          | 0/3 [00:00<?, ?it/s]

Contrast:   0%|          | 0/2 [00:00<?, ?it/s]

MotionBlur:   0%|          | 0/3 [00:00<?, ?it/s]

model             ViT-only  ViT+aug  Fusion  Fusion+aug
corruption level                                       
Clean      0.00      97.25    95.92   97.25       94.83
Contrast   0.40      70.00    91.75   70.67       83.75
           0.60      83.58    94.42   83.25       90.25
Gamma      1.50      96.00    92.33   96.08       91.58
           2.00      92.33    87.58   92.75       85.50
           2.50      88.42    84.08   87.42       81.25
MotionBlur 5.00      85.92    87.83   85.33       88.67
           9.00      81.75    78.50   84.00       81.42
           13.00     74.67    72.92   77.00       75.75
Poisson    15.00     52.08    86.58   54.50       86.75
           30.00     54.33    90.75   63.67       90.92
           60.00     57.75    92.92   76.75       92.42
SaltPepper 0.02      91.00    86.92   58.42       80.67
           0.05      79.75    81.67   50.08       69.33
           0.10      74.83    68.08   50.00       59.50


In [11]:
# CELL 11: READOUT -- mean accuracy on unseen corruptions per model
unseen=df[df.corruption!='Clean']
mean_by_model=unseen.groupby('model')['acc'].mean().reindex(order)
print('Mean accuracy over ALL unseen corruptions:')
print(mean_by_model.round(2).to_string())
print('\nInterpretation guide:')
print('  - If Fusion (no aug) >> ViT-only  -> spectral gives general robustness.')
print('  - If Fusion/Fusion+aug > ViT+aug on unseen -> spectral has a NICHE that')
print('    augmentation does not cover (the key result).')
print('  - If ViT+aug already wins everywhere -> augmentation generalizes; spectral')
print('    niche is weak (still an honest finding).')

Mean accuracy over ALL unseen corruptions:
model
ViT-only      77.32
ViT+aug       85.45
Fusion        73.57
Fusion+aug    82.70

Interpretation guide:
  - If Fusion (no aug) >> ViT-only  -> spectral gives general robustness.
  - If Fusion/Fusion+aug > ViT+aug on unseen -> spectral has a NICHE that
    augmentation does not cover (the key result).
  - If ViT+aug already wins everywhere -> augmentation generalizes; spectral
    niche is weak (still an honest finding).
